# 🍕 Zomato KPT Signal Decontamination Pipeline

## Problem Statement
Zomato's Kitchen Prep Time (KPT) predictions are corrupted by **rider-influenced FOR (Food Order Ready) signals**. When riders arrive early, restaurants mark food as "ready" even when it isn't, leading to noisy training labels and inaccurate predictions.

## Solution
We detect dirty labels using **rider GPS proximity** at FOR timestamp, decontaminate them, and train improved models on clean data.

---

# Stage 1 — Data Ingestion

Load raw synthetic dataset, parse timestamps, validate data types, handle nulls.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import math
import warnings
import os
warnings.filterwarnings('ignore')

# MLflow for experiment tracking
import mlflow
import mlflow.xgboost
import mlflow.sklearn
from mlflow.models.signature import infer_signature

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Setup MLflow
MLFLOW_TRACKING_URI = "../mlruns"
EXPERIMENT_NAME = "Zomato_KPT_Signal_Decontamination"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print("✅ Libraries loaded successfully")
print(f"✅ MLflow tracking URI: {MLFLOW_TRACKING_URI}")
print(f"✅ MLflow experiment: {EXPERIMENT_NAME}")

In [ ]:
# Load the dataset
DATA_PATH = '../Data_Generator/zomato_orders_100k.csv'

df_raw = pd.read_csv(DATA_PATH)
print(f"📊 Loaded {len(df_raw):,} rows and {len(df_raw.columns)} columns")
print(f"\n📋 Columns: {list(df_raw.columns)}")

In [ ]:
# Parse timestamp columns
timestamp_cols = ['order_confirmed_at', 'for_timestamp', 'rider_arrived_at']

for col in timestamp_cols:
    df_raw[col] = pd.to_datetime(df_raw[col])
    print(f"✅ Parsed {col} → dtype: {df_raw[col].dtype}")

# Validate data types
print("\n📋 Data Types:")
print(df_raw.dtypes)

In [ ]:
# Check for nulls
null_counts = df_raw.isnull().sum()
null_pct = (null_counts / len(df_raw) * 100).round(2)

null_summary = pd.DataFrame({
    'Null Count': null_counts,
    'Null %': null_pct
})

print("🔍 Null Value Analysis:")
if null_counts.sum() == 0:
    print("✅ No null values found in any column!")
else:
    print(null_summary[null_summary['Null Count'] > 0])

In [ ]:
# Create working copy
df = df_raw.copy()

# Basic validation
assert df['order_id'].nunique() == len(df), "Order IDs not unique!"
assert (df['for_timestamp'] >= df['order_confirmed_at']).all(), "FOR before order confirmed!"
assert df['actual_kpt_seconds'].between(0, 10000).all(), "KPT values out of range!"

print("✅ All validations passed!")
print(f"\n📊 Dataset Summary:")
print(f"   • Orders: {len(df):,}")
print(f"   • Merchants: {df['merchant_id'].nunique():,}")
print(f"   • Cities: {df['city'].nunique()}")
print(f"   • Date Range: {df['order_confirmed_at'].min().date()} to {df['order_confirmed_at'].max().date()}")

In [ ]:
# Preview the data
df.head(10)

---
# Stage 2 — Exploratory Data Analysis (EDA)

Understand the data deeply. Tell the story of the problem with charts and numbers.

In [ ]:
# 2.1 Distribution of Raw KPT
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['actual_kpt_seconds'] / 60, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(df['actual_kpt_seconds'].mean() / 60, color='red', linestyle='--', label=f"Mean: {df['actual_kpt_seconds'].mean()/60:.1f} min")
axes[0].axvline(df['actual_kpt_seconds'].median() / 60, color='orange', linestyle='--', label=f"Median: {df['actual_kpt_seconds'].median()/60:.1f} min")
axes[0].set_xlabel('KPT (minutes)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Kitchen Prep Time (KPT)')
axes[0].legend()

# Box plot
axes[1].boxplot(df['actual_kpt_seconds'] / 60, vert=True)
axes[1].set_ylabel('KPT (minutes)')
axes[1].set_title('KPT Box Plot')

plt.tight_layout()
plt.show()

print(f"\n📊 KPT Statistics:")
print(f"   Mean: {df['actual_kpt_seconds'].mean()/60:.2f} minutes")
print(f"   Median: {df['actual_kpt_seconds'].median()/60:.2f} minutes")
print(f"   Std Dev: {df['actual_kpt_seconds'].std()/60:.2f} minutes")
print(f"   Min: {df['actual_kpt_seconds'].min()/60:.2f} minutes")
print(f"   Max: {df['actual_kpt_seconds'].max()/60:.2f} minutes")

In [ ]:
# 2.2 FOR Rider-Influenced Analysis
rider_influenced = df['is_for_rider_influenced'].value_counts()
rider_influenced_pct = df['is_for_rider_influenced'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
colors = ['#2ecc71', '#e74c3c']
labels = ['Genuine FOR Signal', 'Rider-Influenced FOR']
axes[0].pie([rider_influenced[False], rider_influenced[True]], 
            labels=labels, autopct='%1.1f%%', colors=colors, explode=(0, 0.05))
axes[0].set_title('FOR Signal Quality Distribution')

# KPT comparison
kpt_clean = df[df['is_for_rider_influenced'] == False]['actual_kpt_seconds'] / 60
kpt_dirty = df[df['is_for_rider_influenced'] == True]['actual_kpt_seconds'] / 60

axes[1].hist(kpt_clean, bins=40, alpha=0.6, label=f'Clean Labels (n={len(kpt_clean):,})', color='green')
axes[1].hist(kpt_dirty, bins=40, alpha=0.6, label=f'Dirty Labels (n={len(kpt_dirty):,})', color='red')
axes[1].set_xlabel('KPT (minutes)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('KPT Distribution: Clean vs Dirty Labels')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\n🚨 THE PROBLEM:")
print(f"   {rider_influenced_pct[True]:.1f}% of FOR signals are rider-influenced (dirty labels)")
print(f"   This means ~{rider_influenced[True]:,} orders have unreliable prep time labels")

In [ ]:
# 2.3 KPT by Cuisine Type
cuisine_kpt = df.groupby('cuisine_type')['actual_kpt_seconds'].agg(['mean', 'median', 'std', 'count'])
cuisine_kpt['mean_min'] = cuisine_kpt['mean'] / 60
cuisine_kpt = cuisine_kpt.sort_values('mean', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(cuisine_kpt.index, cuisine_kpt['mean_min'], color='steelblue', edgecolor='black')
ax.set_xlabel('Average KPT (minutes)')
ax.set_title('Average Kitchen Prep Time by Cuisine Type')

# Add value labels
for bar, val in zip(bars, cuisine_kpt['mean_min']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, f'{val:.1f} min', va='center')

plt.tight_layout()
plt.show()

print("\n📊 KPT by Cuisine Type:")
print(cuisine_kpt[['mean_min', 'count']].round(2).to_string())

In [ ]:
# 2.4 KPT by Time of Day (Peak vs Off-Peak)
df['hour'] = df['order_confirmed_at'].dt.hour

hourly_kpt = df.groupby('hour')['actual_kpt_seconds'].mean() / 60

fig, ax = plt.subplots(figsize=(14, 6))

colors = ['#e74c3c' if (12 <= h < 14) or (19 <= h < 22) else '#3498db' for h in hourly_kpt.index]
bars = ax.bar(hourly_kpt.index, hourly_kpt.values, color=colors, edgecolor='black')

ax.axhline(hourly_kpt.mean(), color='black', linestyle='--', label=f'Overall Avg: {hourly_kpt.mean():.1f} min')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Average KPT (minutes)')
ax.set_title('Average KPT by Hour of Day (Red = Peak Hours)')
ax.set_xticks(range(24))
ax.legend()

plt.tight_layout()
plt.show()

peak_kpt = df[df['is_peak_hour'] == True]['actual_kpt_seconds'].mean() / 60
offpeak_kpt = df[df['is_peak_hour'] == False]['actual_kpt_seconds'].mean() / 60

print(f"\n⏰ Peak Hour Analysis:")
print(f"   Peak Hours (12-2pm, 7-10pm): {peak_kpt:.2f} min avg KPT")
print(f"   Off-Peak Hours: {offpeak_kpt:.2f} min avg KPT")
print(f"   Difference: {peak_kpt - offpeak_kpt:.2f} minutes longer during peak")

In [ ]:
# 2.5 KPT by City
city_kpt = df.groupby('city')['actual_kpt_seconds'].agg(['mean', 'median', 'std', 'count'])
city_kpt['mean_min'] = city_kpt['mean'] / 60
city_kpt = city_kpt.sort_values('mean', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
axes[0].barh(city_kpt.index, city_kpt['mean_min'], color='teal', edgecolor='black')
axes[0].set_xlabel('Average KPT (minutes)')
axes[0].set_title('Average KPT by City')

# Order volume
axes[1].barh(city_kpt.index, city_kpt['count'], color='coral', edgecolor='black')
axes[1].set_xlabel('Number of Orders')
axes[1].set_title('Order Volume by City')

plt.tight_layout()
plt.show()

print("\n🏙️ KPT by City:")
print(city_kpt[['mean_min', 'count']].round(2).to_string())

In [ ]:
# 2.6 Kitchen Load vs KPT Correlation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Concurrent orders vs KPT
concurrent_kpt = df.groupby('concurrent_orders')['actual_kpt_seconds'].mean() / 60
axes[0].plot(concurrent_kpt.index, concurrent_kpt.values, marker='o', linewidth=2, markersize=8)
axes[0].set_xlabel('Concurrent Orders in Kitchen')
axes[0].set_ylabel('Average KPT (minutes)')
axes[0].set_title('KPT vs Concurrent Orders')
axes[0].set_xticks(range(1, 9))

# Kitchen load index vs KPT
df['load_bin'] = pd.cut(df['kitchen_load_index'], bins=10)
load_kpt = df.groupby('load_bin')['actual_kpt_seconds'].mean() / 60
axes[1].bar(range(len(load_kpt)), load_kpt.values, color='purple', alpha=0.7)
axes[1].set_xlabel('Kitchen Load Index (bins)')
axes[1].set_ylabel('Average KPT (minutes)')
axes[1].set_title('KPT vs Kitchen Load Index')

plt.tight_layout()
plt.show()

# Correlation
corr_concurrent = df['concurrent_orders'].corr(df['actual_kpt_seconds'])
corr_load = df['kitchen_load_index'].corr(df['actual_kpt_seconds'])

print(f"\n📈 Correlations with KPT:")
print(f"   Concurrent Orders: r = {corr_concurrent:.3f}")
print(f"   Kitchen Load Index: r = {corr_load:.3f}")

In [ ]:
# 2.7 Merchant Behavior Patterns (Gaming Detection)
merchant_dirty_rate = df.groupby('merchant_id').agg({
    'is_for_rider_influenced': 'mean',
    'order_id': 'count',
    'actual_kpt_seconds': 'mean'
}).rename(columns={
    'is_for_rider_influenced': 'dirty_rate',
    'order_id': 'order_count',
    'actual_kpt_seconds': 'avg_kpt'
})

merchant_dirty_rate['avg_kpt_min'] = merchant_dirty_rate['avg_kpt'] / 60

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of dirty rates
axes[0].hist(merchant_dirty_rate['dirty_rate'] * 100, bins=30, edgecolor='black', color='crimson', alpha=0.7)
axes[0].axvline(35, color='black', linestyle='--', label='Expected 35%')
axes[0].set_xlabel('Dirty Label Rate (%)')
axes[0].set_ylabel('Number of Merchants')
axes[0].set_title('Distribution of Dirty Label Rates Across Merchants')
axes[0].legend()

# Scatter: Dirty rate vs Avg KPT
axes[1].scatter(merchant_dirty_rate['dirty_rate'] * 100, merchant_dirty_rate['avg_kpt_min'], 
                alpha=0.5, s=merchant_dirty_rate['order_count']/2)
axes[1].set_xlabel('Dirty Label Rate (%)')
axes[1].set_ylabel('Average KPT (minutes)')
axes[1].set_title('Merchant Dirty Rate vs Average KPT (bubble size = order volume)')

plt.tight_layout()
plt.show()

# Identify worst offenders
high_gamers = merchant_dirty_rate[merchant_dirty_rate['dirty_rate'] > 0.5]
print(f"\n🚨 Merchant Gaming Analysis:")
print(f"   Merchants with >50% dirty labels: {len(high_gamers)} ({len(high_gamers)/len(merchant_dirty_rate)*100:.1f}%)")
print(f"   Average dirty rate across all merchants: {merchant_dirty_rate['dirty_rate'].mean()*100:.1f}%")

### 📊 EDA Summary

**Key Findings:**
1. **~35% of FOR signals are rider-influenced** — a significant data quality issue
2. **KPT varies by cuisine** — Biryani and complex meals take longer
3. **Peak hours have higher KPT** — 12-2pm and 7-10pm show elevated prep times
4. **Kitchen load correlates with KPT** — busier kitchens = longer prep times
5. **Some merchants have abnormally high dirty rates** — potential systematic gaming

---

# Stage 3 — Signal Audit (Core of Solution)

Formally identify dirty labels using rider GPS data. Calculate distance between rider and restaurant at FOR timestamp.

In [ ]:
# Haversine distance function
def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate the great circle distance between two points 
    on the earth (specified in decimal degrees).
    Returns distance in meters.
    """
    R = 6371000  # Earth's radius in meters
    
    lat1_rad = np.radians(lat1)
    lat2_rad = np.radians(lat2)
    delta_lat = np.radians(lat2 - lat1)
    delta_lon = np.radians(lon2 - lon1)
    
    a = np.sin(delta_lat/2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(delta_lon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    
    return R * c

print("✅ Haversine distance function defined")

In [ ]:
# Calculate rider distance from restaurant at FOR timestamp
df['rider_distance_at_FOR'] = haversine_distance(
    df['restaurant_lat'], df['restaurant_lng'],
    df['rider_lat'], df['rider_lng']
)

print(f"📏 Rider Distance at FOR Statistics:")
print(f"   Min: {df['rider_distance_at_FOR'].min():.1f} meters")
print(f"   Max: {df['rider_distance_at_FOR'].max():.1f} meters")
print(f"   Mean: {df['rider_distance_at_FOR'].mean():.1f} meters")
print(f"   Median: {df['rider_distance_at_FOR'].median():.1f} meters")

In [ ]:
# Define thresholds and categorize labels
DIRTY_THRESHOLD = 200      # meters - definitely suspicious
BORDERLINE_THRESHOLD = 400  # meters - needs probabilistic treatment

def categorize_label(distance):
    if distance <= DIRTY_THRESHOLD:
        return 'dirty'
    elif distance <= BORDERLINE_THRESHOLD:
        return 'borderline'
    else:
        return 'clean'

df['label_category'] = df['rider_distance_at_FOR'].apply(categorize_label)
df['is_label_dirty'] = df['rider_distance_at_FOR'] <= DIRTY_THRESHOLD

# Summary
label_dist = df['label_category'].value_counts()
label_pct = df['label_category'].value_counts(normalize=True) * 100

print(f"\n📊 Label Categorization (Threshold: {DIRTY_THRESHOLD}m):")
print(f"   Clean (>{BORDERLINE_THRESHOLD}m): {label_dist.get('clean', 0):,} ({label_pct.get('clean', 0):.1f}%)")
print(f"   Borderline ({DIRTY_THRESHOLD}-{BORDERLINE_THRESHOLD}m): {label_dist.get('borderline', 0):,} ({label_pct.get('borderline', 0):.1f}%)")
print(f"   Dirty (≤{DIRTY_THRESHOLD}m): {label_dist.get('dirty', 0):,} ({label_pct.get('dirty', 0):.1f}%)")

In [ ]:
# Visualize distance distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of distances
axes[0].hist(df['rider_distance_at_FOR'], bins=100, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(DIRTY_THRESHOLD, color='red', linestyle='--', linewidth=2, label=f'Dirty Threshold ({DIRTY_THRESHOLD}m)')
axes[0].axvline(BORDERLINE_THRESHOLD, color='orange', linestyle='--', linewidth=2, label=f'Borderline ({BORDERLINE_THRESHOLD}m)')
axes[0].set_xlabel('Rider Distance from Restaurant (meters)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Rider Distance at FOR Timestamp')
axes[0].legend()
axes[0].set_xlim(0, 3500)

# Category pie chart
colors = {'clean': '#2ecc71', 'borderline': '#f39c12', 'dirty': '#e74c3c'}
axes[1].pie(label_dist, labels=label_dist.index, autopct='%1.1f%%', 
            colors=[colors[l] for l in label_dist.index], explode=[0.02]*len(label_dist))
axes[1].set_title('Label Quality Categories')

plt.tight_layout()
plt.show()

In [ ]:
# Quantify the damage: KPT comparison across label categories
kpt_by_category = df.groupby('label_category')['actual_kpt_seconds'].agg(['mean', 'median', 'std', 'count'])
kpt_by_category['mean_min'] = kpt_by_category['mean'] / 60
kpt_by_category['median_min'] = kpt_by_category['median'] / 60

print("📊 KPT by Label Category:")
print(kpt_by_category[['mean_min', 'median_min', 'count']].round(2).to_string())

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plots by category
category_order = ['clean', 'borderline', 'dirty']
df_plot = df.copy()
df_plot['kpt_min'] = df_plot['actual_kpt_seconds'] / 60

bp = axes[0].boxplot([df_plot[df_plot['label_category'] == cat]['kpt_min'] for cat in category_order],
                     labels=category_order, patch_artist=True)
for patch, color in zip(bp['boxes'], [colors['clean'], colors['borderline'], colors['dirty']]):
    patch.set_facecolor(color)
axes[0].set_xlabel('Label Category')
axes[0].set_ylabel('KPT (minutes)')
axes[0].set_title('KPT Distribution by Label Category')

# Mean comparison
means = [kpt_by_category.loc[cat, 'mean_min'] for cat in category_order if cat in kpt_by_category.index]
bars = axes[1].bar(category_order[:len(means)], means, 
                   color=[colors[cat] for cat in category_order[:len(means)]], edgecolor='black')
axes[1].set_xlabel('Label Category')
axes[1].set_ylabel('Mean KPT (minutes)')
axes[1].set_title('Mean KPT by Label Category')

for bar, val in zip(bars, means):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, 
                 f'{val:.1f} min', ha='center')

plt.tight_layout()
plt.show()

# Calculate the noise impact
clean_mean = kpt_by_category.loc['clean', 'mean'] if 'clean' in kpt_by_category.index else 0
dirty_mean = kpt_by_category.loc['dirty', 'mean'] if 'dirty' in kpt_by_category.index else 0

print(f"\n🚨 NOISE QUANTIFICATION:")
print(f"   Clean label avg KPT: {clean_mean/60:.2f} minutes")
print(f"   Dirty label avg KPT: {dirty_mean/60:.2f} minutes")
print(f"   Difference: {abs(clean_mean - dirty_mean)/60:.2f} minutes")

In [ ]:
# Verify against ground truth flag
confusion = pd.crosstab(df['is_label_dirty'], df['is_for_rider_influenced'], 
                        rownames=['GPS Detected'], colnames=['Ground Truth'])
print("\n🔍 GPS Detection vs Ground Truth Confusion Matrix:")
print(confusion)

accuracy = (df['is_label_dirty'] == df['is_for_rider_influenced']).mean() * 100
print(f"\n✅ Detection Accuracy: {accuracy:.2f}%")

---
# Stage 4 — Label Decontamination

Fix dirty labels using three strategies:
- **Strategy A**: Hard Filter (remove dirty rows)
- **Strategy B**: Median Imputation (replace with historical median)
- **Strategy C**: Confidence Weighting (keep all, weight by confidence)

In [ ]:
# Extract hour slot for aggregation
df['hour_slot'] = df['order_confirmed_at'].dt.hour

# Strategy A: Hard Filter (remove dirty labels)
df_strategy_a = df[df['is_label_dirty'] == False].copy()
print(f"Strategy A - Hard Filter:")
print(f"   Original rows: {len(df):,}")
print(f"   After filtering: {len(df_strategy_a):,}")
print(f"   Data loss: {(1 - len(df_strategy_a)/len(df))*100:.1f}%")

In [ ]:
# Strategy B: Median Imputation
# For each dirty label, use the same merchant's historical clean KPT for same hour slot and cuisine

# Calculate clean medians by merchant, hour, cuisine
clean_df = df[df['is_label_dirty'] == False]

# Merchant-Hour-Cuisine median
merchant_hour_cuisine_median = clean_df.groupby(
    ['merchant_id', 'hour_slot', 'cuisine_type']
)['actual_kpt_seconds'].median().reset_index()
merchant_hour_cuisine_median.columns = ['merchant_id', 'hour_slot', 'cuisine_type', 'imputed_kpt_mhc']

# Merchant-Hour median (fallback)
merchant_hour_median = clean_df.groupby(
    ['merchant_id', 'hour_slot']
)['actual_kpt_seconds'].median().reset_index()
merchant_hour_median.columns = ['merchant_id', 'hour_slot', 'imputed_kpt_mh']

# Merchant median (fallback 2)
merchant_median = clean_df.groupby('merchant_id')['actual_kpt_seconds'].median().reset_index()
merchant_median.columns = ['merchant_id', 'imputed_kpt_m']

# Global median (final fallback)
global_median = clean_df['actual_kpt_seconds'].median()

print(f"✅ Imputation lookup tables created")
print(f"   Merchant-Hour-Cuisine combinations: {len(merchant_hour_cuisine_median):,}")
print(f"   Merchant-Hour combinations: {len(merchant_hour_median):,}")
print(f"   Global median: {global_median/60:.2f} minutes")

In [ ]:
# Apply Strategy B imputation
df_strategy_b = df.copy()

# Merge lookup tables
df_strategy_b = df_strategy_b.merge(merchant_hour_cuisine_median, 
                                     on=['merchant_id', 'hour_slot', 'cuisine_type'], 
                                     how='left')
df_strategy_b = df_strategy_b.merge(merchant_hour_median, 
                                     on=['merchant_id', 'hour_slot'], 
                                     how='left')
df_strategy_b = df_strategy_b.merge(merchant_median, 
                                     on='merchant_id', 
                                     how='left')

# Apply imputation with fallback chain
def impute_kpt(row):
    if not row['is_label_dirty']:
        return row['actual_kpt_seconds']  # Clean label, keep original
    
    # Try each fallback level
    if pd.notna(row['imputed_kpt_mhc']):
        return row['imputed_kpt_mhc']
    elif pd.notna(row['imputed_kpt_mh']):
        return row['imputed_kpt_mh']
    elif pd.notna(row['imputed_kpt_m']):
        return row['imputed_kpt_m']
    else:
        return global_median

df_strategy_b['clean_kpt_seconds'] = df_strategy_b.apply(impute_kpt, axis=1)

# Clean up temporary columns
df_strategy_b = df_strategy_b.drop(columns=['imputed_kpt_mhc', 'imputed_kpt_mh', 'imputed_kpt_m'])

print(f"Strategy B - Median Imputation:")
print(f"   Rows retained: {len(df_strategy_b):,}")
print(f"   Dirty labels imputed: {df_strategy_b['is_label_dirty'].sum():,}")
print(f"   Original mean KPT: {df_strategy_b['actual_kpt_seconds'].mean()/60:.2f} min")
print(f"   Cleaned mean KPT: {df_strategy_b['clean_kpt_seconds'].mean()/60:.2f} min")

In [ ]:
# Strategy C: Confidence Weighting
df_strategy_c = df.copy()

# Assign confidence weights
def assign_confidence(row):
    if row['label_category'] == 'clean':
        return 1.0
    elif row['label_category'] == 'borderline':
        return 0.6
    else:  # dirty
        return 0.3

df_strategy_c['sample_weight'] = df_strategy_c.apply(assign_confidence, axis=1)
df_strategy_c['clean_kpt_seconds'] = df_strategy_c['actual_kpt_seconds']  # Keep original, use weights

print(f"Strategy C - Confidence Weighting:")
print(f"   Rows retained: {len(df_strategy_c):,}")
print(f"   Weight distribution:")
print(df_strategy_c['sample_weight'].value_counts().sort_index())

In [ ]:
# Compare strategies vs ground truth
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Original (dirty) distribution
axes[0, 0].hist(df['actual_kpt_seconds']/60, bins=50, alpha=0.7, color='red', label='All Data (Dirty)')
axes[0, 0].set_xlabel('KPT (minutes)')
axes[0, 0].set_title('Original Data Distribution')
axes[0, 0].axvline(df['actual_kpt_seconds'].mean()/60, color='black', linestyle='--')

# Strategy A
axes[0, 1].hist(df_strategy_a['actual_kpt_seconds']/60, bins=50, alpha=0.7, color='green', label='After Hard Filter')
axes[0, 1].set_xlabel('KPT (minutes)')
axes[0, 1].set_title(f'Strategy A: Hard Filter (n={len(df_strategy_a):,})')
axes[0, 1].axvline(df_strategy_a['actual_kpt_seconds'].mean()/60, color='black', linestyle='--')

# Strategy B
axes[1, 0].hist(df_strategy_b['clean_kpt_seconds']/60, bins=50, alpha=0.7, color='blue', label='After Imputation')
axes[1, 0].set_xlabel('KPT (minutes)')
axes[1, 0].set_title(f'Strategy B: Median Imputation (n={len(df_strategy_b):,})')
axes[1, 0].axvline(df_strategy_b['clean_kpt_seconds'].mean()/60, color='black', linestyle='--')

# Clean labels only (ground truth comparison)
clean_only = df[df['is_for_rider_influenced'] == False]['actual_kpt_seconds']/60
axes[1, 1].hist(clean_only, bins=50, alpha=0.7, color='purple', label='Ground Truth Clean')
axes[1, 1].set_xlabel('KPT (minutes)')
axes[1, 1].set_title(f'Ground Truth Clean Labels (n={len(clean_only):,})')
axes[1, 1].axvline(clean_only.mean(), color='black', linestyle='--')

plt.tight_layout()
plt.show()

# Summary comparison
print("\n📊 Strategy Comparison:")
print(f"{'Strategy':<25} {'Rows':<12} {'Mean KPT':<15} {'Std Dev':<12}")
print("-" * 65)
print(f"{'Original (Dirty)':<25} {len(df):<12,} {df['actual_kpt_seconds'].mean()/60:<15.2f} {df['actual_kpt_seconds'].std()/60:<12.2f}")
print(f"{'A: Hard Filter':<25} {len(df_strategy_a):<12,} {df_strategy_a['actual_kpt_seconds'].mean()/60:<15.2f} {df_strategy_a['actual_kpt_seconds'].std()/60:<12.2f}")
print(f"{'B: Median Imputation':<25} {len(df_strategy_b):<12,} {df_strategy_b['clean_kpt_seconds'].mean()/60:<15.2f} {df_strategy_b['clean_kpt_seconds'].std()/60:<12.2f}")
print(f"{'Ground Truth Clean':<25} {len(clean_only):<12,} {clean_only.mean():<15.2f} {clean_only.std():<12.2f}")

In [ ]:
# Select best strategy - using Strategy B for main pipeline
# (keeps all data while correcting dirty labels)

df_clean = df_strategy_b.copy()
print("✅ Selected Strategy B (Median Imputation) for the main pipeline")
print(f"   Clean KPT column: 'clean_kpt_seconds'")
print(f"   This will be the training target for the improved model")

---
# Stage 5 — Feature Engineering

Build all input signals that will feed the prediction model.

In [ ]:
# 5.1 Time-based Features
df_clean['hour_of_day'] = df_clean['order_confirmed_at'].dt.hour
df_clean['day_of_week_num'] = df_clean['order_confirmed_at'].dt.dayofweek  # 0=Monday, 6=Sunday
df_clean['month'] = df_clean['order_confirmed_at'].dt.month

# Is festival day (simplified - major Indian festivals)
festival_dates = [
    '2023-01-26', '2023-03-08', '2023-04-14', '2023-08-15', '2023-10-24', '2023-11-12', '2023-12-25',
    '2024-01-26', '2024-03-25', '2024-04-14', '2024-08-15', '2024-10-12', '2024-11-01', '2024-12-25'
]
festival_dates = pd.to_datetime(festival_dates)
df_clean['is_festival_day'] = df_clean['order_confirmed_at'].dt.date.isin(festival_dates.date)

print("✅ Time-based features created")
print(f"   hour_of_day, day_of_week_num, month, is_festival_day, is_peak_hour, is_weekend")

In [ ]:
# 5.2 Merchant-based Features (using clean labels only)

# Merchant average KPT (all clean orders)
merchant_avg_kpt = clean_df.groupby('merchant_id')['actual_kpt_seconds'].mean().reset_index()
merchant_avg_kpt.columns = ['merchant_id', 'merchant_avg_kpt']

# Merchant KPT std (consistency measure)
merchant_std_kpt = clean_df.groupby('merchant_id')['actual_kpt_seconds'].std().fillna(0).reset_index()
merchant_std_kpt.columns = ['merchant_id', 'merchant_kpt_std']

# Merchant average KPT by hour slot
merchant_hour_kpt = clean_df.groupby(['merchant_id', 'hour_slot'])['actual_kpt_seconds'].mean().reset_index()
merchant_hour_kpt.columns = ['merchant_id', 'hour_slot', 'merchant_hour_avg_kpt']

# Merchant dirty label rate (how often does this merchant game)
merchant_dirty_rate = df.groupby('merchant_id')['is_label_dirty'].mean().reset_index()
merchant_dirty_rate.columns = ['merchant_id', 'merchant_dirty_rate']

# Merge features
df_clean = df_clean.merge(merchant_avg_kpt, on='merchant_id', how='left')
df_clean = df_clean.merge(merchant_std_kpt, on='merchant_id', how='left')
df_clean = df_clean.merge(merchant_hour_kpt, on=['merchant_id', 'hour_slot'], how='left')
df_clean = df_clean.merge(merchant_dirty_rate, on='merchant_id', how='left')

# Fill missing with global values
df_clean['merchant_avg_kpt'] = df_clean['merchant_avg_kpt'].fillna(clean_df['actual_kpt_seconds'].mean())
df_clean['merchant_kpt_std'] = df_clean['merchant_kpt_std'].fillna(clean_df['actual_kpt_seconds'].std())
df_clean['merchant_hour_avg_kpt'] = df_clean['merchant_hour_avg_kpt'].fillna(df_clean['merchant_avg_kpt'])
df_clean['merchant_dirty_rate'] = df_clean['merchant_dirty_rate'].fillna(0.35)  # Overall average

print("✅ Merchant-based features created")
print(f"   merchant_avg_kpt, merchant_kpt_std, merchant_hour_avg_kpt, merchant_dirty_rate")

In [ ]:
# 5.3 Order Complexity Features

# Item category complexity weights
category_weights = {
    'Beverage': 0.3,
    'Snacks': 0.5,
    'Dessert': 0.6,
    'Main Course': 1.0,
    'Biryani': 1.4,
    'Combo': 1.6
}

df_clean['category_weight'] = df_clean['item_category'].map(category_weights)

# Total complexity score = num_items * item_complexity_score * category_weight
df_clean['total_complexity_score'] = (df_clean['num_items'] * 
                                       df_clean['item_complexity_score'] * 
                                       df_clean['category_weight'])

print("✅ Order complexity features created")
print(f"   category_weight, total_complexity_score")
print(f"   Total complexity range: {df_clean['total_complexity_score'].min():.2f} - {df_clean['total_complexity_score'].max():.2f}")

In [ ]:
# 5.4 Encode Categorical Features
from sklearn.preprocessing import LabelEncoder

# City encoding
city_encoder = LabelEncoder()
df_clean['city_encoded'] = city_encoder.fit_transform(df_clean['city'])

# Weather encoding
weather_encoder = LabelEncoder()
df_clean['weather_encoded'] = weather_encoder.fit_transform(df_clean['weather_condition'])

# Cuisine encoding
cuisine_encoder = LabelEncoder()
df_clean['cuisine_encoded'] = cuisine_encoder.fit_transform(df_clean['cuisine_type'])

# Item category encoding
category_encoder = LabelEncoder()
df_clean['item_category_encoded'] = category_encoder.fit_transform(df_clean['item_category'])

print("✅ Categorical features encoded")
print(f"   city_encoded: {dict(zip(city_encoder.classes_, range(len(city_encoder.classes_))))}")
print(f"   weather_encoded: {dict(zip(weather_encoder.classes_, range(len(weather_encoder.classes_))))}")

In [ ]:
# 5.5 Define Feature Matrix
FEATURE_COLUMNS = [
    # Time features
    'hour_of_day',
    'day_of_week_num',
    'month',
    'is_peak_hour',
    'is_weekend',
    'is_festival_day',
    
    # Merchant features
    'merchant_avg_kpt',
    'merchant_kpt_std',
    'merchant_hour_avg_kpt',
    'merchant_dirty_rate',
    
    # Order complexity features
    'num_items',
    'item_complexity_score',
    'category_weight',
    'total_complexity_score',
    
    # Kitchen load features
    'concurrent_orders',
    'kitchen_load_index',
    
    # Encoded categoricals
    'city_encoded',
    'weather_encoded',
    'cuisine_encoded',
    'item_category_encoded',
]

# Convert booleans to int
for col in ['is_peak_hour', 'is_weekend', 'is_festival_day']:
    df_clean[col] = df_clean[col].astype(int)

print(f"✅ Feature matrix defined with {len(FEATURE_COLUMNS)} features:")
for i, col in enumerate(FEATURE_COLUMNS, 1):
    print(f"   {i:2d}. {col}")

In [ ]:
# Verify no missing values in features
missing = df_clean[FEATURE_COLUMNS].isnull().sum()
if missing.sum() == 0:
    print("✅ No missing values in feature matrix")
else:
    print("⚠️ Missing values found:")
    print(missing[missing > 0])

---
# Stage 6 — Baseline Model (Dirty Labels)

Train a model on the original noisy data exactly as Zomato does today. This is our baseline.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb

# Prepare data
X = df_clean[FEATURE_COLUMNS]
y_dirty = df_clean['actual_kpt_seconds']  # Original dirty labels
y_clean = df_clean['clean_kpt_seconds']   # Decontaminated labels

# Train-test split (same split for fair comparison)
X_train, X_test, y_dirty_train, y_dirty_test, y_clean_train, y_clean_test = train_test_split(
    X, y_dirty, y_clean, test_size=0.2, random_state=42
)

print(f"📊 Data Split:")
print(f"   Training set: {len(X_train):,} rows")
print(f"   Test set: {len(X_test):,} rows")

In [ ]:
# Helper function for model evaluation
def evaluate_model(y_true, y_pred, model_name):
    """Calculate and display model metrics."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    errors = np.abs(y_true - y_pred)
    p50_error = np.percentile(errors, 50)
    p90_error = np.percentile(errors, 90)
    
    metrics = {
        'MAE (seconds)': mae,
        'MAE (minutes)': mae / 60,
        'RMSE (seconds)': rmse,
        'RMSE (minutes)': rmse / 60,
        'P50 Error (seconds)': p50_error,
        'P90 Error (seconds)': p90_error,
    }
    
    print(f"\n📊 {model_name} Performance:")
    print(f"   MAE: {mae:.1f} seconds ({mae/60:.2f} minutes)")
    print(f"   RMSE: {rmse:.1f} seconds ({rmse/60:.2f} minutes)")
    print(f"   P50 Error: {p50_error:.1f} seconds ({p50_error/60:.2f} minutes)")
    print(f"   P90 Error: {p90_error:.1f} seconds ({p90_error/60:.2f} minutes)")
    
    return metrics


def log_model_to_mlflow(model, model_name, metrics, X_train, y_train, params, tags=None):
    """Log model, metrics, and artifacts to MLflow."""
    with mlflow.start_run(run_name=model_name):
        # Log parameters
        mlflow.log_params(params)
        
        # Log metrics
        mlflow.log_metric("mae_seconds", metrics['MAE (seconds)'])
        mlflow.log_metric("mae_minutes", metrics['MAE (minutes)'])
        mlflow.log_metric("rmse_seconds", metrics['RMSE (seconds)'])
        mlflow.log_metric("rmse_minutes", metrics['RMSE (minutes)'])
        mlflow.log_metric("p50_error_seconds", metrics['P50 Error (seconds)'])
        mlflow.log_metric("p90_error_seconds", metrics['P90 Error (seconds)'])
        
        # Log tags
        if tags:
            mlflow.set_tags(tags)
        
        # Create model signature
        signature = infer_signature(X_train, model.predict(X_train))
        
        # Log model
        mlflow.xgboost.log_model(
            model, 
            artifact_path="model",
            signature=signature,
            registered_model_name=f"KPT_{model_name.replace(' ', '_')}"
        )
        
        # Log feature importance as artifact
        feature_importance = pd.DataFrame({
            'feature': FEATURE_COLUMNS,
            'importance': model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        feature_importance.to_csv(f"/tmp/{model_name}_feature_importance.csv", index=False)
        mlflow.log_artifact(f"/tmp/{model_name}_feature_importance.csv")
        
        run_id = mlflow.active_run().info.run_id
        print(f"   ✅ Model logged to MLflow (Run ID: {run_id})")
        
        return run_id

print("✅ MLflow helper functions defined")

In [ ]:
# Train Baseline Model on DIRTY labels
print("🔄 Training Baseline Model (on dirty labels)...")

# Model parameters
baseline_params = {
    'n_estimators': 200,
    'max_depth': 6,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'label_type': 'dirty'
}

baseline_model = xgb.XGBRegressor(
    n_estimators=baseline_params['n_estimators'],
    max_depth=baseline_params['max_depth'],
    learning_rate=baseline_params['learning_rate'],
    subsample=baseline_params['subsample'],
    colsample_bytree=baseline_params['colsample_bytree'],
    random_state=baseline_params['random_state'],
    n_jobs=-1
)

baseline_model.fit(X_train, y_dirty_train, 
                   eval_set=[(X_test, y_dirty_test)],
                   verbose=False)

print("✅ Baseline model trained!")

# Predictions
y_pred_baseline = baseline_model.predict(X_test)

# Evaluate against ACTUAL KPT (ground truth)
baseline_metrics = evaluate_model(y_dirty_test, y_pred_baseline, "Baseline Model (Dirty Labels)")

# Log to MLflow
baseline_run_id = log_model_to_mlflow(
    model=baseline_model,
    model_name="Baseline_Dirty_Labels",
    metrics=baseline_metrics,
    X_train=X_train,
    y_train=y_dirty_train,
    params=baseline_params,
    tags={
        "model_type": "XGBoost",
        "label_strategy": "dirty",
        "stage": "baseline"
    }
)

In [ ]:
# Feature Importance - Baseline Model
feature_importance = pd.DataFrame({
    'feature': FEATURE_COLUMNS,
    'importance': baseline_model.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(feature_importance['feature'], feature_importance['importance'], color='steelblue')
ax.set_xlabel('Importance')
ax.set_title('Feature Importance - Baseline Model')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

---
# Stage 7 — Improved Model (Clean Labels)

Same model architecture, same features, but trained on decontaminated clean KPT labels.

In [ ]:
# Train Improved Model on CLEAN labels
print("🔄 Training Improved Model (on clean labels)...")

# Model parameters
improved_params = {
    'n_estimators': 200,
    'max_depth': 6,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'label_type': 'clean',
    'decontamination_strategy': 'median_imputation'
}

improved_model = xgb.XGBRegressor(
    n_estimators=improved_params['n_estimators'],
    max_depth=improved_params['max_depth'],
    learning_rate=improved_params['learning_rate'],
    subsample=improved_params['subsample'],
    colsample_bytree=improved_params['colsample_bytree'],
    random_state=improved_params['random_state'],
    n_jobs=-1
)

improved_model.fit(X_train, y_clean_train,
                   eval_set=[(X_test, y_clean_test)],
                   verbose=False)

print("✅ Improved model trained!")

# Predictions
y_pred_improved = improved_model.predict(X_test)

# Evaluate against CLEAN KPT (decontaminated ground truth)
improved_metrics = evaluate_model(y_clean_test, y_pred_improved, "Improved Model (Clean Labels)")

# Log to MLflow
improved_run_id = log_model_to_mlflow(
    model=improved_model,
    model_name="Improved_Clean_Labels",
    metrics=improved_metrics,
    X_train=X_train,
    y_train=y_clean_train,
    params=improved_params,
    tags={
        "model_type": "XGBoost",
        "label_strategy": "clean",
        "stage": "improved",
        "decontamination": "median_imputation"
    }
)

In [ ]:
# Also train with Strategy C (Confidence Weighting)
print("\n🔄 Training Weighted Model (Strategy C)...")

# Get sample weights
weights_train = df_clean.loc[X_train.index, 'sample_weight'].values

# Model parameters
weighted_params = {
    'n_estimators': 200,
    'max_depth': 6,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'label_type': 'dirty_weighted',
    'decontamination_strategy': 'confidence_weighting',
    'clean_weight': 1.0,
    'borderline_weight': 0.6,
    'dirty_weight': 0.3
}

weighted_model = xgb.XGBRegressor(
    n_estimators=weighted_params['n_estimators'],
    max_depth=weighted_params['max_depth'],
    learning_rate=weighted_params['learning_rate'],
    subsample=weighted_params['subsample'],
    colsample_bytree=weighted_params['colsample_bytree'],
    random_state=weighted_params['random_state'],
    n_jobs=-1
)

weighted_model.fit(X_train, y_dirty_train,  # Original labels but weighted
                   sample_weight=weights_train,
                   eval_set=[(X_test, y_dirty_test)],
                   verbose=False)

print("✅ Weighted model trained!")

y_pred_weighted = weighted_model.predict(X_test)
weighted_metrics = evaluate_model(y_dirty_test, y_pred_weighted, "Weighted Model (Strategy C)")

# Log to MLflow
weighted_run_id = log_model_to_mlflow(
    model=weighted_model,
    model_name="Weighted_Confidence",
    metrics=weighted_metrics,
    X_train=X_train,
    y_train=y_dirty_train,
    params=weighted_params,
    tags={
        "model_type": "XGBoost",
        "label_strategy": "weighted",
        "stage": "improved",
        "decontamination": "confidence_weighting"
    }
)

In [ ]:
# MLflow Model Comparison Dashboard
print("\\n" + "="*80)
print("📊 MLFLOW MODEL REGISTRY")
print("="*80)

# Get all runs from the experiment
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

# Display comparison table
comparison_cols = ['run_id', 'tags.mlflow.runName', 'metrics.mae_seconds', 
                   'metrics.rmse_seconds', 'metrics.p50_error_seconds', 'metrics.p90_error_seconds']
available_cols = [col for col in comparison_cols if col in runs.columns]

print("\\n📋 All Logged Runs:")
print(runs[available_cols].to_string())

# Find best model by MAE
if 'metrics.mae_seconds' in runs.columns:
    best_run = runs.loc[runs['metrics.mae_seconds'].idxmin()]
    print(f"\\n🏆 Best Model (by MAE): {best_run.get('tags.mlflow.runName', 'N/A')}")
    print(f"   MAE: {best_run['metrics.mae_seconds']:.1f} seconds")
    print(f"   Run ID: {best_run['run_id']}")

print(f"\\n🌐 To view MLflow UI, run in terminal:")
print(f"   cd {os.path.abspath('../')} && mlflow ui --port 5000")
print(f"   Then open: http://localhost:5000")

### 🔬 MLflow Experiment Tracking

All models are automatically logged to MLflow with:
- **Parameters**: Hyperparameters (n_estimators, max_depth, learning_rate, etc.)
- **Metrics**: MAE, RMSE, P50/P90 errors
- **Artifacts**: Feature importance CSVs, trained models
- **Model Registry**: Versioned models ready for deployment

To view the MLflow dashboard:
```bash
mlflow ui --backend-store-uri ../mlruns --port 5000
```
Then open http://localhost:5000 in your browser.

---
# Stage 8 — Evaluation and Comparison

Compare dirty model vs clean model side by side and show the delta.

In [ ]:
# Create comparison table
comparison_data = {
    'Metric': ['MAE (seconds)', 'MAE (minutes)', 'RMSE (seconds)', 'P50 Error (sec)', 'P90 Error (sec)'],
    'Baseline (Dirty)': [
        baseline_metrics['MAE (seconds)'],
        baseline_metrics['MAE (minutes)'],
        baseline_metrics['RMSE (seconds)'],
        baseline_metrics['P50 Error (seconds)'],
        baseline_metrics['P90 Error (seconds)']
    ],
    'Improved (Clean)': [
        improved_metrics['MAE (seconds)'],
        improved_metrics['MAE (minutes)'],
        improved_metrics['RMSE (seconds)'],
        improved_metrics['P50 Error (seconds)'],
        improved_metrics['P90 Error (seconds)']
    ],
    'Weighted (Strategy C)': [
        weighted_metrics['MAE (seconds)'],
        weighted_metrics['MAE (minutes)'],
        weighted_metrics['RMSE (seconds)'],
        weighted_metrics['P50 Error (seconds)'],
        weighted_metrics['P90 Error (seconds)']
    ]
}

comparison_df = pd.DataFrame(comparison_data)
comparison_df['Improvement (B vs I)'] = comparison_df['Baseline (Dirty)'] - comparison_df['Improved (Clean)']
comparison_df['Improvement %'] = ((comparison_df['Baseline (Dirty)'] - comparison_df['Improved (Clean)']) / comparison_df['Baseline (Dirty)'] * 100).round(1)

print("\n" + "="*80)
print("📊 MODEL COMPARISON")
print("="*80)
print(comparison_df.to_string(index=False))

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# MAE comparison
models = ['Baseline\n(Dirty)', 'Improved\n(Clean)', 'Weighted\n(Strategy C)']
maes = [baseline_metrics['MAE (minutes)'], improved_metrics['MAE (minutes)'], weighted_metrics['MAE (minutes)']]
colors = ['#e74c3c', '#2ecc71', '#3498db']

bars = axes[0].bar(models, maes, color=colors, edgecolor='black')
axes[0].set_ylabel('MAE (minutes)')
axes[0].set_title('Mean Absolute Error Comparison')
for bar, val in zip(bars, maes):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
                 f'{val:.2f}', ha='center', fontsize=10)

# P90 error comparison
p90s = [baseline_metrics['P90 Error (seconds)']/60, improved_metrics['P90 Error (seconds)']/60, weighted_metrics['P90 Error (seconds)']/60]
bars = axes[1].bar(models, p90s, color=colors, edgecolor='black')
axes[1].set_ylabel('P90 Error (minutes)')
axes[1].set_title('90th Percentile Error Comparison')
for bar, val in zip(bars, p90s):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
                 f'{val:.2f}', ha='center', fontsize=10)

# Prediction distribution
axes[2].hist(y_pred_baseline/60, bins=50, alpha=0.5, label='Baseline', color='red')
axes[2].hist(y_pred_improved/60, bins=50, alpha=0.5, label='Improved', color='green')
axes[2].set_xlabel('Predicted KPT (minutes)')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Prediction Distribution')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Segment-level analysis
test_df = df_clean.loc[X_test.index].copy()
test_df['pred_baseline'] = y_pred_baseline
test_df['pred_improved'] = y_pred_improved
test_df['error_baseline'] = np.abs(test_df['actual_kpt_seconds'] - test_df['pred_baseline'])
test_df['error_improved'] = np.abs(test_df['clean_kpt_seconds'] - test_df['pred_improved'])

# By cuisine type
print("\n📊 Improvement by Cuisine Type:")
cuisine_comparison = test_df.groupby('cuisine_type').agg({
    'error_baseline': 'mean',
    'error_improved': 'mean'
}).round(1)
cuisine_comparison['improvement_sec'] = cuisine_comparison['error_baseline'] - cuisine_comparison['error_improved']
print(cuisine_comparison.to_string())

# By city
print("\n📊 Improvement by City:")
city_comparison = test_df.groupby('city').agg({
    'error_baseline': 'mean',
    'error_improved': 'mean'
}).round(1)
city_comparison['improvement_sec'] = city_comparison['error_baseline'] - city_comparison['error_improved']
print(city_comparison.to_string())

# By peak hour
print("\n📊 Improvement by Peak Hour:")
peak_comparison = test_df.groupby('is_peak_hour').agg({
    'error_baseline': 'mean',
    'error_improved': 'mean'
}).round(1)
peak_comparison['improvement_sec'] = peak_comparison['error_baseline'] - peak_comparison['error_improved']
print(peak_comparison.to_string())

In [ ]:
# High-dirty-rate merchants vs low-dirty-rate merchants
median_dirty_rate = test_df['merchant_dirty_rate'].median()

high_dirty_merchants = test_df[test_df['merchant_dirty_rate'] > median_dirty_rate]
low_dirty_merchants = test_df[test_df['merchant_dirty_rate'] <= median_dirty_rate]

print("\n📊 Improvement for High vs Low Dirty-Rate Merchants:")
print(f"\nHigh Dirty-Rate Merchants (>{median_dirty_rate*100:.1f}%):")
print(f"   Baseline MAE: {high_dirty_merchants['error_baseline'].mean():.1f} sec")
print(f"   Improved MAE: {high_dirty_merchants['error_improved'].mean():.1f} sec")
print(f"   Improvement: {high_dirty_merchants['error_baseline'].mean() - high_dirty_merchants['error_improved'].mean():.1f} sec")

print(f"\nLow Dirty-Rate Merchants (≤{median_dirty_rate*100:.1f}%):")
print(f"   Baseline MAE: {low_dirty_merchants['error_baseline'].mean():.1f} sec")
print(f"   Improved MAE: {low_dirty_merchants['error_improved'].mean():.1f} sec")
print(f"   Improvement: {low_dirty_merchants['error_baseline'].mean() - low_dirty_merchants['error_improved'].mean():.1f} sec")

---
# Stage 9 — Business Impact Translation

Convert model accuracy improvement into real business numbers.

In [ ]:
# Business parameters (Zomato scale estimates)
DAILY_ORDERS = 5_000_000  # 5 million orders per day
RIDER_COST_PER_MINUTE = 2  # INR per minute of idle time
CUSTOMER_CHURN_RATE_PER_LATE_ORDER = 0.02  # 2% chance of losing customer on very late delivery
AVERAGE_ORDER_VALUE = 350  # INR
CUSTOMER_LIFETIME_VALUE = 15000  # INR

# Calculate improvements
mae_improvement_seconds = baseline_metrics['MAE (seconds)'] - improved_metrics['MAE (seconds)']
mae_improvement_minutes = mae_improvement_seconds / 60

print("="*80)
print("💰 BUSINESS IMPACT ANALYSIS")
print("="*80)

print(f"\n📊 Model Improvement:")
print(f"   MAE Reduction: {mae_improvement_seconds:.1f} seconds ({mae_improvement_minutes:.2f} minutes) per order")

In [ ]:
# Impact 1: Rider Wait Time Reduction
daily_rider_minutes_saved = DAILY_ORDERS * mae_improvement_minutes
daily_rider_cost_saved = daily_rider_minutes_saved * RIDER_COST_PER_MINUTE
annual_rider_cost_saved = daily_rider_cost_saved * 365

print(f"\n🏍️ Rider Wait Time Reduction:")
print(f"   Minutes saved per day: {daily_rider_minutes_saved:,.0f}")
print(f"   Hours saved per day: {daily_rider_minutes_saved/60:,.0f}")
print(f"   Daily cost savings: ₹{daily_rider_cost_saved:,.0f}")
print(f"   Annual cost savings: ₹{annual_rider_cost_saved/10000000:.2f} Crore")

In [ ]:
# Impact 2: ETA Accuracy Improvement
p90_improvement = baseline_metrics['P90 Error (seconds)'] - improved_metrics['P90 Error (seconds)']

# Calculate orders with significantly reduced wait (more than 5 min improvement)
significant_improvement_threshold = 300  # 5 minutes in seconds
error_reduction = test_df['error_baseline'] - test_df['error_improved']
pct_significantly_improved = (error_reduction > significant_improvement_threshold).mean() * 100

print(f"\n⏱️ ETA Accuracy Improvement:")
print(f"   P90 Error Reduction: {p90_improvement:.1f} seconds ({p90_improvement/60:.2f} minutes)")
print(f"   Orders with >5 min improvement: {pct_significantly_improved:.1f}%")

In [ ]:
# Impact 3: Late Delivery Reduction
# Define "late" as prediction error > 10 minutes
late_threshold = 600  # 10 minutes in seconds

late_baseline = (test_df['error_baseline'] > late_threshold).mean() * 100
late_improved = (test_df['error_improved'] > late_threshold).mean() * 100
late_reduction = late_baseline - late_improved

daily_late_orders_prevented = DAILY_ORDERS * (late_reduction / 100)
potential_churned_customers_saved = daily_late_orders_prevented * CUSTOMER_CHURN_RATE_PER_LATE_ORDER
ltv_saved_daily = potential_churned_customers_saved * CUSTOMER_LIFETIME_VALUE

print(f"\n🚚 Late Delivery Reduction (>10 min error):")
print(f"   Baseline late rate: {late_baseline:.2f}%")
print(f"   Improved late rate: {late_improved:.2f}%")
print(f"   Reduction: {late_reduction:.2f} percentage points")
print(f"   Daily late orders prevented: {daily_late_orders_prevented:,.0f}")
print(f"   Customers saved from churning (daily): {potential_churned_customers_saved:,.0f}")
print(f"   Daily LTV protected: ₹{ltv_saved_daily:,.0f}")

In [ ]:
# Summary Business Case
print("\n" + "="*80)
print("📈 EXECUTIVE SUMMARY - ANNUAL BUSINESS IMPACT")
print("="*80)

annual_savings = annual_rider_cost_saved + (ltv_saved_daily * 365)

print(f"""
┌─────────────────────────────────────────────────────────────────┐
│  KPT SIGNAL DECONTAMINATION - ROI SUMMARY                       │
├─────────────────────────────────────────────────────────────────┤
│  Model Improvement:                                             │
│    • MAE reduced by {mae_improvement_minutes:.2f} minutes per order                    │
│    • P90 error reduced by {p90_improvement/60:.2f} minutes                          │
│    • Late delivery rate reduced by {late_reduction:.2f}%                       │
├─────────────────────────────────────────────────────────────────┤
│  Financial Impact (Annual):                                     │
│    • Rider efficiency savings: ₹{annual_rider_cost_saved/10000000:.2f} Crore                      │
│    • Customer retention value: ₹{(ltv_saved_daily * 365)/10000000:.2f} Crore                    │
│    • TOTAL ANNUAL IMPACT: ₹{annual_savings/10000000:.2f} Crore                        │
├─────────────────────────────────────────────────────────────────┤
│  Operational Impact (Daily):                                    │
│    • {daily_rider_minutes_saved/60:,.0f} rider-hours saved                                │
│    • {daily_late_orders_prevented:,.0f} late deliveries prevented                        │
│    • {potential_churned_customers_saved:,.0f} customers retained                               │
└─────────────────────────────────────────────────────────────────┘
""")

---
# Stage 10 — Scalability Design

How this solution works at Zomato's actual scale of 300,000 merchants in real time.

In [ ]:
# System Architecture Diagram (ASCII)
architecture = """
╔══════════════════════════════════════════════════════════════════════════════════════╗
║                     ZOMATO KPT PREDICTION - SCALABLE ARCHITECTURE                     ║
╠══════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                       ║
║    ┌─────────────┐     ┌─────────────┐     ┌─────────────┐     ┌─────────────┐       ║
║    │   ZOMATO    │     │   RIDER     │     │  MERCHANT   │     │   WEATHER   │       ║
║    │    APP      │     │    APP      │     │    APP      │     │    API      │       ║
║    └──────┬──────┘     └──────┬──────┘     └──────┬──────┘     └──────┬──────┘       ║
║           │                   │                   │                   │              ║
║           └───────────────────┴───────────────────┴───────────────────┘              ║
║                                       │                                              ║
║                                       ▼                                              ║
║                          ┌────────────────────────┐                                  ║
║                          │      KAFKA STREAM      │                                  ║
║                          │   (Event Ingestion)    │                                  ║
║                          └───────────┬────────────┘                                  ║
║                                      │                                               ║
║           ┌──────────────────────────┼──────────────────────────┐                    ║
║           ▼                          ▼                          ▼                    ║
║  ┌─────────────────┐      ┌─────────────────────┐      ┌─────────────────┐          ║
║  │  FOR EVENT      │      │   GPS DISTANCE      │      │  FEATURE        │          ║
║  │  PROCESSOR      │──────│   CALCULATOR        │──────│  ENRICHMENT     │          ║
║  │                 │      │   (Haversine)       │      │  SERVICE        │          ║
║  └────────┬────────┘      └─────────────────────┘      └────────┬────────┘          ║
║           │                                                      │                   ║
║           │         ┌─────────────────────────────┐              │                   ║
║           │         │     DIRTY LABEL DETECTOR    │              │                   ║
║           └────────▶│     (Distance < 200m?)      │◀─────────────┘                   ║
║                     └──────────────┬──────────────┘                                  ║
║                                    │                                                 ║
║                                    ▼                                                 ║
║                     ┌─────────────────────────────┐                                  ║
║                     │     LABEL DECONTAMINATOR    │                                  ║
║                     │  (Median Imputation Engine) │                                  ║
║                     └──────────────┬──────────────┘                                  ║
║                                    │                                                 ║
║           ┌────────────────────────┴────────────────────────┐                        ║
║           ▼                                                 ▼                        ║
║  ┌─────────────────────┐                        ┌─────────────────────┐             ║
║  │   FEATURE STORE     │                        │   KPT PREDICTION    │             ║
║  │  ┌───────────────┐  │                        │      MODEL          │             ║
║  │  │ Merchant KPT  │  │                        │   (XGBoost/API)     │             ║
║  │  │ Historical    │  │──────────────────────▶│                     │             ║
║  │  │ Medians       │  │                        │  Response: <100ms  │             ║
║  │  └───────────────┘  │                        └──────────┬──────────┘             ║
║  │  ┌───────────────┐  │                                   │                        ║
║  │  │ Hour-Slot     │  │                                   ▼                        ║
║  │  │ Aggregates    │  │                        ┌─────────────────────┐             ║
║  │  └───────────────┘  │                        │   ETA CALCULATION   │             ║
║  │  ┌───────────────┐  │                        │      SERVICE        │             ║
║  │  │ Dirty Rate    │  │                        └──────────┬──────────┘             ║
║  │  │ per Merchant  │  │                                   │                        ║
║  │  └───────────────┘  │                                   ▼                        ║
║  │  (Redis/DynamoDB)   │                        ┌─────────────────────┐             ║
║  └─────────────────────┘                        │   CUSTOMER APP      │             ║
║                                                 │   RIDER APP         │             ║
║           │                                     └─────────────────────┘             ║
║           │                                                                          ║
║           ▼                                                                          ║
║  ┌─────────────────────┐                        ┌─────────────────────┐             ║
║  │   BATCH PIPELINE    │                        │    MONITORING       │             ║
║  │   (Hourly Refresh)  │                        │    DASHBOARD        │             ║
║  │                     │                        │                     │             ║
║  │  • Update medians   │                        │  • Dirty label %    │             ║
║  │  • Recalculate      │                        │  • Model accuracy   │             ║
║  │    dirty rates      │                        │  • Merchant alerts  │             ║
║  │  • Retrain model    │                        │  • Business KPIs    │             ║
║  │    (weekly)         │                        │                     │             ║
║  └─────────────────────┘                        └─────────────────────┘             ║
║                                                                                       ║
╚══════════════════════════════════════════════════════════════════════════════════════╝
"""

print(architecture)

In [ ]:
# Scalability specifications
scalability_specs = """
╔══════════════════════════════════════════════════════════════════════════════════════╗
║                          SCALABILITY SPECIFICATIONS                                   ║
╠══════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                       ║
║  📊 SCALE REQUIREMENTS:                                                               ║
║     • 300,000 active merchants                                                        ║
║     • 5 million orders per day                                                        ║
║     • ~60 orders per second peak                                                      ║
║     • 10 million rider GPS pings per hour                                             ║
║                                                                                       ║
║  ⚡ REAL-TIME COMPONENTS:                                                              ║
║                                                                                       ║
║     1. GPS Distance Calculator                                                        ║
║        • Input: Rider GPS + Merchant coordinates                                      ║
║        • Computation: Haversine formula (O(1))                                        ║
║        • Latency: <5ms                                                                ║
║        • Tech: In-memory, stateless microservice                                      ║
║                                                                                       ║
║     2. Dirty Label Detector                                                           ║
║        • Input: Distance at FOR timestamp                                             ║
║        • Logic: distance < 200m → dirty                                               ║
║        • Latency: <1ms                                                                ║
║        • Tech: Simple threshold check                                                 ║
║                                                                                       ║
║     3. Feature Enrichment Service                                                     ║
║        • Input: Order metadata                                                        ║
║        • Lookups: Merchant stats from Feature Store                                   ║
║        • Latency: <20ms (Redis lookup)                                                ║
║        • Tech: Redis Cluster with replication                                         ║
║                                                                                       ║
║     4. KPT Prediction Model                                                           ║
║        • Input: 20 features                                                           ║
║        • Model: XGBoost (serialized)                                                  ║
║        • Latency: <50ms                                                               ║
║        • Tech: Model served via FastAPI on Kubernetes                                 ║
║                                                                                       ║
║     TOTAL END-TO-END LATENCY: <100ms                                                  ║
║                                                                                       ║
║  🔄 BATCH COMPONENTS (Hourly/Daily):                                                  ║
║                                                                                       ║
║     1. Feature Store Refresh (Hourly)                                                 ║
║        • Recalculate merchant-hour-cuisine medians                                    ║
║        • Update dirty rates per merchant                                              ║
║        • Tech: Spark on EMR → Redis                                                   ║
║                                                                                       ║
║     2. Model Retraining (Weekly)                                                      ║
║        • Pull last 30 days of clean labels                                            ║
║        • Retrain XGBoost with new data                                                ║
║        • A/B test before full deployment                                              ║
║        • Tech: SageMaker / Kubeflow                                                   ║
║                                                                                       ║
║  💾 DATA STORES:                                                                       ║
║                                                                                       ║
║     • Feature Store: Redis Cluster (10 nodes, 100GB)                                  ║
║       - Key: merchant_id:hour_slot                                                    ║
║       - Value: {median_kpt, std_kpt, dirty_rate}                                      ║
║       - TTL: 2 hours                                                                  ║
║                                                                                       ║
║     • Raw Events: Kafka (30 day retention)                                            ║
║     • Historical Data: S3 + Athena                                                    ║
║     • Model Registry: MLflow                                                          ║
║                                                                                       ║
╚══════════════════════════════════════════════════════════════════════════════════════╝
"""

print(scalability_specs)

In [ ]:
# Pseudo-code for real-time processing
realtime_code = '''
# Real-time FOR Event Processing (Python pseudo-code)

class FOREventProcessor:
    """
    Processes Food Order Ready (FOR) events in real-time.
    Detects dirty labels and returns clean KPT prediction.
    """
    
    def __init__(self):
        self.feature_store = RedisClient("feature-store-cluster")
        self.model = load_xgboost_model("kpt_model_v2.json")
        self.merchant_coords = load_merchant_coordinates()  # Preloaded
    
    async def process_for_event(self, event: FOREvent) -> KPTPrediction:
        """
        Main entry point. Processes FOR event and returns KPT prediction.
        Target latency: <100ms
        """
        
        # Step 1: Calculate rider distance (< 5ms)
        restaurant_coords = self.merchant_coords[event.merchant_id]
        distance = haversine_distance(
            restaurant_coords.lat, restaurant_coords.lng,
            event.rider_lat, event.rider_lng
        )
        
        # Step 2: Detect dirty label (< 1ms)
        is_dirty = distance < 200  # meters
        
        # Step 3: Get merchant features from Feature Store (< 20ms)
        merchant_features = await self.feature_store.get(
            key=f"{event.merchant_id}:{event.hour_slot}"
        )
        
        # Step 4: Build feature vector (< 5ms)
        features = self.build_features(
            event=event,
            merchant_features=merchant_features,
            distance=distance,
            is_dirty=is_dirty
        )
        
        # Step 5: Model prediction (< 50ms)
        predicted_kpt = self.model.predict(features)
        
        # Step 6: Apply correction if dirty label
        if is_dirty and merchant_features:
            # Use historical median instead of raw prediction
            corrected_kpt = merchant_features.median_kpt
            confidence = 0.6
        else:
            corrected_kpt = predicted_kpt
            confidence = 0.9
        
        return KPTPrediction(
            order_id=event.order_id,
            predicted_kpt_seconds=corrected_kpt,
            is_dirty_label=is_dirty,
            rider_distance_meters=distance,
            confidence=confidence
        )
'''

print(realtime_code)

---
# 📋 Pipeline Summary

## What We Built

| Stage | Description | Output |
|-------|-------------|--------|
| 1. Data Ingestion | Load and validate raw dataset | Clean DataFrame with parsed timestamps |
| 2. EDA | Understand data distributions and patterns | Visualizations proving the problem exists |
| 3. Signal Audit | Detect dirty labels using GPS distance | `is_label_dirty` column, `rider_distance_at_FOR` |
| 4. Label Decontamination | Fix dirty labels with median imputation | `clean_kpt_seconds` column |
| 5. Feature Engineering | Build 20 input features | Feature matrix ready for modeling |
| 6. Baseline Model | Train on dirty labels | Baseline performance metrics |
| 7. Improved Model | Train on clean labels | Improved performance metrics |
| 8. Evaluation | Compare models side-by-side | Proof of improvement |
| 9. Business Impact | Translate to ₹ savings | ROI and business case |
| 10. Scalability Design | Real-time architecture | System design for production |

## Key Results

- **35% of FOR signals** were identified as rider-influenced (dirty labels)
- **Model accuracy improved** after training on decontaminated labels
- **Business impact** quantified in crores of annual savings
- **Scalable architecture** designed for 5M orders/day

---

In [ ]:
# Save processed data for future use
df_clean.to_csv('../Data_Generator/zomato_orders_processed.csv', index=False)
print("✅ Processed dataset saved to ../Data_Generator/zomato_orders_processed.csv")

# Save models locally
import joblib
joblib.dump(baseline_model, '../Data_Generator/baseline_model.joblib')
joblib.dump(improved_model, '../Data_Generator/improved_model.joblib')
joblib.dump(weighted_model, '../Data_Generator/weighted_model.joblib')
print("✅ Models saved to ../Data_Generator/")

# Save encoders for inference
encoders = {
    'city_encoder': city_encoder,
    'weather_encoder': weather_encoder,
    'cuisine_encoder': cuisine_encoder,
    'category_encoder': category_encoder,
    'feature_columns': FEATURE_COLUMNS
}
joblib.dump(encoders, '../Data_Generator/encoders.joblib')
print("✅ Encoders saved for inference")

print("\\n" + "="*80)
print("📊 MLFLOW SUMMARY")
print("="*80)
print(f"\\n🔬 Experiment: {EXPERIMENT_NAME}")
print(f"📁 Tracking URI: {os.path.abspath(MLFLOW_TRACKING_URI)}")
print(f"\\n📋 Logged Models:")
print(f"   1. Baseline_Dirty_Labels (Run ID: {baseline_run_id[:8]}...)")
print(f"   2. Improved_Clean_Labels (Run ID: {improved_run_id[:8]}...)")
print(f"   3. Weighted_Confidence (Run ID: {weighted_run_id[:8]}...)")
print(f"\\n🚀 To launch MLflow UI:")
print(f"   mlflow ui --backend-store-uri {os.path.abspath(MLFLOW_TRACKING_URI)} --port 5000")
print(f"\\n   Then open: http://localhost:5000")